# Multimodal Media Search with Gemini Embedding 2.0

This notebook demonstrates how to build a multimodal search engine (Images, Videos) using the Gemini Embedding 2.0 model.

## Phase 1: Environment Setup

In this phase, we will install the necessary libraries and authenticate with Google Cloud.

In [ ]:
# Install necessary libraries
%pip install google-genai google-cloud-aiplatform Pillow opencv-python numpy ipywidgets pyOpenSSL

# Import libraries
import os
import time
from google import genai
from google.genai import types
from google.cloud import aiplatform
from PIL import Image
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, Image as IPImage, Video as IPVideo

print("Libraries imported successfully.")

### Authentication

To use the Gemini API, you need to authenticate. If you are running this in Google Colab, you can use the user authentication. If you are running locally, make sure you have set the `GEMINI_API_KEY` environment variable or have the correct credentials configured.

In [ ]:
# Replace with your actual Gemini API key
# %env GEMINI_API_KEY='YOUR_GEMINI_API_KEY'
print("Please set your GEMINI_API_KEY environment variable or uncomment the line above to set it.")

In [ ]:
# Initialize the GenAI client
# The client will automatically look for the GEMINI_API_KEY environment variable.
PROJECT_ID = "YOUR_PROJECT_ID" # Replace with your project ID
LOCATION = "us-central1" # Replace with your region if different
BUCKET_NAME = "ai-multimodal-data" # Cloud Storage bucket for data

try:
    client = genai.Client(
        vertexai=True,
        project=PROJECT_ID,
        location=LOCATION
    )
    # Initialize Vertex AI SDK for Vector Search
    aiplatform.init(project=PROJECT_ID, location=LOCATION)
    print("GenAI Client and Vertex AI SDK initialized successfully.")
except Exception as e:
    print(f"Error initializing client: {e}")
    print("Please make sure GEMINI_API_KEY is set or you are authenticated.")

# Define the model ID
MODEL_ID = 'gemini-embedding-2-preview'
print(f"Using model: {MODEL_ID}")

## Phase 2: Loading Data from Cloud Storage

In this phase, we will check our authentication by accessing the Cloud Storage bucket and previewing some of the existing data (images and videos) that we will use for our search engine.

In [ ]:
# Check Authentication and Preview Data from GCS
from google.cloud import storage
from PIL import Image
import io
import ipywidgets as widgets
from IPython.display import display, Image as IPImage, HTML
import os
import traceback
import base64

print(f"Checking access to bucket: {BUCKET_NAME}...")
try:
    storage_client = storage.Client(project=PROJECT_ID)
    bucket = storage_client.bucket(BUCKET_NAME)
    
    # List blobs without limit to see the full count
    # Note: For extremely large buckets, this might take a moment.
    blobs = list(bucket.list_blobs())
    print(f"Found {len(blobs)} files in bucket.")
    
    print("\nPreviewing sample files:")
    
    images = []
    videos = []
    
    for blob in blobs:
        if blob.name.endswith(('.png', '.jpg')):
            images.append(blob)
        elif blob.name.endswith('.mp4'):
            videos.append(blob)
            
    print(f"Total Images found: {len(images)}")
    print(f"Total Videos found: {len(videos)}")
            
    # Display up to 3 images
    img_widgets = []
    print("\nImage Previews:")
    for blob in images[:3]:
        print(f"- {blob.name} ({blob.content_type})")
        image_bytes = blob.download_as_bytes()
        out = widgets.Output()
        with out:
            display(IPImage(data=image_bytes, width=200))
        img_widgets.append(out)
        
    if img_widgets:
        display(widgets.HBox(img_widgets))
        
    # Video preview skipped as requested because they are too long
    if videos:
        print(f"\nFound {len(videos)} videos. Skipping preview to avoid downloading large files.")
        
    if not images and not videos:
        print("No image or video files found in the bucket.")
            
except Exception as e:
    print(f"Error accessing bucket: {e}")
    traceback.print_exc()
    print("Please ensure your credentials are correct and you have access to the bucket.")

## Phase 2b: Advanced Video Processing (Chunking)

To handle long videos (40-60 minutes), we need to segment them into smaller chunks (e.g., 10 seconds) to capture specific semantic moments and avoid diluting embeddings.

### OpenCV vs. MoviePy for Video Chunking

We are using **OpenCV** as preferred, but here is a quick comparison:

*   **OpenCV**: Fast, low-level control, already installed. *Con*: Does not handle audio (resulting chunks will be silent).
*   **MoviePy**: Easy API (`clip.subclip`), preserves audio. *Con*: Slower, needs installation.

We will use the video `gs://ai-multimodal-data/Eurasia_KBS_130903.mp4` for testing.

In [ ]:
import os
import cv2
from google.cloud import storage

# Configuration
VIDEO_URI = "gs://ai-multimodal-data/Eurasia_KBS_130903.mp4"
OUTPUT_BUCKET = BUCKET_NAME # Using bucket defined in cell 4
OUTPUT_FOLDER = "vid-chunks"
CHUNK_DURATION = 10  # seconds

# Download the video file locally for processing
def download_blob(bucket_name, source_blob_name, destination_file_name):
    """Downloads a blob from the bucket."""
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(source_blob_name)
    blob.download_to_filename(destination_file_name)
    print(f"Blob {source_blob_name} downloaded to {destination_file_name}.")

# Extract bucket and blob name from URI
src_bucket_name = "ai-multimodal-data"
src_blob_name = "Eurasia_KBS_130903.mp4"
local_video_path = "Eurasia_KBS_130903.mp4"

# Download if not exists
if not os.path.exists(local_video_path):
    print(f"Downloading {VIDEO_URI}...")
    try:
        download_blob(src_bucket_name, src_blob_name, local_video_path)
    except Exception as e:
        print(f"Error downloading video: {e}")
        print("Please ensure you have access to the source bucket.")
else:
    print(f"Local file {local_video_path} already exists.")

# Chunking function using OpenCV
def chunk_video_opencv(video_path, chunk_duration_sec=10, output_dir="chunks"):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total_frames / fps
    
    print(f"Video FPS: {fps}, Total Frames: {total_frames}, Duration: {duration:.2f}s")
    
    frames_per_chunk = int(fps * chunk_duration_sec)
    
    metadata = []
    current_chunk = 0
    frame_count = 0
    out = None
    
    while True:
        ret, frame = cap.read()
        if not ret:
            break
            
        if frame_count % frames_per_chunk == 0:
            if out is not None:
                out.release()
                
            chunk_file = os.path.join(output_dir, f"chunk_{current_chunk}.mp4")
            
            # Try avc1 (H.264) for browser compatibility
            fourcc = cv2.VideoWriter_fourcc(*'avc1')
            out = cv2.VideoWriter(chunk_file, fourcc, fps, (frame.shape[1], frame.shape[0]))
            
            # Fallback if avc1 fails
            if not out.isOpened():
                print(f"Warning: avc1 codec not supported for {chunk_file}, falling back to mp4v. Video may not play in browser.")
                fourcc = cv2.VideoWriter_fourcc(*'mp4v')
                out = cv2.VideoWriter(chunk_file, fourcc, fps, (frame.shape[1], frame.shape[0]))
            
            start_time = current_chunk * chunk_duration_sec
            end_time = min((current_chunk + 1) * chunk_duration_sec, duration)
            
            metadata.append({
                "chunk_id": current_chunk,
                "file_path": chunk_file,
                "start_time": start_time,
                "end_time": end_time
            })
            current_chunk += 1
            
        out.write(frame)
        frame_count += 1
        
    if out is not None:
        out.release()
        
    cap.release()
    print(f"Created {current_chunk} chunks.")
    return metadata

# Run chunking
if os.path.exists(local_video_path):
    print("Starting video chunking...")
    chunk_metadata = chunk_video_opencv(local_video_path, CHUNK_DURATION, "local_chunks")
    print(f"Generated metadata for {len(chunk_metadata)} chunks.")
    print("Sample metadata:", chunk_metadata[0] if chunk_metadata else "None")
else:
    print("Skipping chunking as video was not downloaded.")

In [ ]:
def upload_directory(local_path, bucket_name, gcs_folder):
    """Uploads a directory to GCS."""
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    
    print(f"Uploading chunks to gs://{bucket_name}/{gcs_folder}...")
    count = 0
    for root, dirs, files in os.walk(local_path):
        for file in files:
            local_file = os.path.join(root, file)
            remote_path = os.path.join(gcs_folder, file)
            blob = bucket.blob(remote_path)
            blob.upload_from_filename(local_file)
            count += 1
            
    print(f"Uploaded {count} files.")

# Upload chunks if they were created
if 'chunk_metadata' in locals() and chunk_metadata:
    try:
        upload_directory("local_chunks", OUTPUT_BUCKET, OUTPUT_FOLDER)
    except Exception as e:
        print(f"Error uploading to GCS: {e}")
        print("Please ensure you have write permissions to the bucket.")
else:
    print("No chunks to upload.")

## Phase 3: Generating Embeddings

In this phase, we will define a function to generate multimodal embeddings using the `gemini-embedding-2-preview` model and process our data pool.

In [ ]:
def generate_multimodal_embedding(content_path, content_type):
    """Generates a multimodal embedding using Gemini Embedding 2.0.
    
    Args:
        content_path: Path to the file (local or GCS gs://) or the text string itself.
        content_type: 'text', 'image', 'video', or 'audio'.
    """
    try:
        if content_type == 'text':
            response = client.models.embed_content(
                model=MODEL_ID,
                contents=content_path
            )
        else:
            # Check if it's a GCS URI
            if isinstance(content_path, str) and content_path.startswith("gs://"):
                mime_type = 'image/jpeg' # Default for images
                if content_type == 'image' and content_path.endswith('.png'):
                    mime_type = 'image/png'
                elif content_type == 'video':
                    mime_type = 'video/mp4'
                elif content_type == 'audio':
                    mime_type = 'audio/mpeg'
                    
                part = types.Part.from_uri(
                    file_uri=content_path,
                    mime_type=mime_type
                )
                contents = part
            else:
                # Handle local files as before
                if content_type == 'image':
                    contents = Image.open(content_path)
                elif content_type == 'video':
                    with open(content_path, 'rb') as f:
                        video_bytes = f.read()
                    contents = types.Part.from_bytes(
                        data=video_bytes,
                        mime_type='video/mp4'
                    )
                elif content_type == 'audio':
                    with open(content_path, 'rb') as f:
                        audio_bytes = f.read()
                    contents = types.Part.from_bytes(
                        data=audio_bytes,
                        mime_type='audio/mpeg'
                    )
                else:
                    raise ValueError(f"Unsupported content type: {content_type}")
            
            response = client.models.embed_content(
                model=MODEL_ID,
                contents=contents
            )
            
        return response.embeddings[0].values
        
    except Exception as e:
        print(f"Error generating embedding for {content_path}: {e}")
        return None

# Registry to store embeddings locally for visualization
embeddings_registry = []

print("Embedding function defined with GCS URI support.")

## Phase 2c: Hybrid Search Architecture

In this phase, we implement a hybrid search that combines:
1.  **Dense Retrieval**: Using `gemini-embedding-2-preview` (multimodal embeddings).
2.  **Sparse Retrieval**: Using BM25 on text descriptions of the video chunks.

We will use Gemini 2.0 Flash to generate descriptions for each chunk to populate the sparse index.

In [ ]:
def generate_chunk_description(video_path):
    """Generates a short description for a video chunk using Gemini 2.5 Flash."""
    try:
        # Load video file
        with open(video_path, 'rb') as f:
            video_bytes = f.read()
            
        part = types.Part.from_bytes(
            data=video_bytes,
            mime_type='video/mp4'
        )
        
        # Using gemini-2.5-flash as requested by user
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[
                part,
                "Provide a short, detailed description of what is happening in this 10-second video clip. Focus on keywords, actions, and objects."
            ]
        )
        return response.text
    except Exception as e:
        print(f"Error generating description for {video_path}: {e}")
        return ""

# Generate descriptions for chunks
# Limit to first 5 chunks for demonstration speed in the workshop
LIMIT = 5
if 'chunk_metadata' in locals() and chunk_metadata:
    print(f"Generating descriptions for the first {LIMIT} chunks using Gemini 2.5 Flash...")
    for item in chunk_metadata[:LIMIT]:
        local_path = item['file_path']
        print(f"Processing {local_path}...")
        description = generate_chunk_description(local_path)
        item['description'] = description
        print(f"Desc: {description.strip()[:100]}...")
else:
    print("No chunk metadata available. Skipping description generation.")

In [ ]:
import math
from collections import Counter

class SimpleBM25:
    """A simple implementation of BM25 for demonstration purposes."""
    def __init__(self, corpus, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus_size = len(corpus)
        self.avgdl = sum(len(doc) for doc in corpus) / self.corpus_size if self.corpus_size > 0 else 0
        self.doc_freqs = []
        self.idf = {}
        self.doc_len = []
        
        # Compute frequencies and lengths
        for doc in corpus:
            self.doc_len.append(len(doc))
            self.doc_freqs.append(Counter(doc))
            
        # Compute IDF
        for doc in corpus:
            for word in set(doc):
                self.idf[word] = self.idf.get(word, 0) + 1
                
        for word, freq in self.idf.items():
            # Standard BM25 IDF formula
            self.idf[word] = math.log((self.corpus_size - freq + 0.5) / (freq + 0.5) + 1)
            
    def get_scores(self, query):
        scores = []
        for i in range(self.corpus_size):
            score = 0
            doc_freq = self.doc_freqs[i]
            doc_len = self.doc_len[i]
            for word in query:
                if word in doc_freq:
                    idf = self.idf.get(word, 0)
                    freq = doc_freq[word]
                    # BM25 formula
                    numerator = freq * (self.k1 + 1)
                    denominator = freq + self.k1 * (1 - self.b + self.b * doc_len / self.avgdl)
                    score += idf * numerator / denominator
            scores.append(score)
        return scores

# Simple tokenization function
def tokenize(text):
    # Convert to lowercase and split by whitespace
    return text.lower().split()

# Prepare corpus for BM25
corpus = []
tokenized_corpus = []
valid_chunks = []

if 'chunk_metadata' in locals() and chunk_metadata:
    for item in chunk_metadata:
        if 'description' in item and item['description']:
            corpus.append(item['description'])
            tokenized_corpus.append(tokenize(item['description']))
            valid_chunks.append(item)
            
    if tokenized_corpus:
        bm25 = SimpleBM25(tokenized_corpus)
        print(f"Custom BM25 index created with {len(corpus)} documents.")
    else:
        print("No descriptions available for BM25 indexing. Please run the description generation cell first.")
else:
    print("No chunk metadata available.")

In [ ]:
# Generate dense embeddings for the chunks
# We use the function defined in Phase 3 (Cell 11)
if 'valid_chunks' in locals() and valid_chunks:
    print("Generating dense embeddings for chunks...")
    for item in valid_chunks:
        local_path = item['file_path']
        print(f"Embedding {local_path}...")
        # generate_multimodal_embedding handles local files
        embedding = generate_multimodal_embedding(local_path, 'video')
        item['dense_embedding'] = embedding
    print(f"Dense embeddings generated for {len(valid_chunks)} chunks.")
else:
    print("No valid chunks available. Please run the previous cells.")

In [ ]:
import numpy as np
import base64
from IPython.display import display, HTML

def hybrid_search(query_text, alpha=0.7, top_k=5):
    """Performs hybrid search combining dense and sparse results.
    
    Args:
        query_text: The search query string.
        alpha: Weight for dense results (0.7 favors dense, 0.6 favors sparse).
        top_k: Number of results to return.
    """
    if 'valid_chunks' not in locals() or not valid_chunks:
        print("No indexed chunks available.")
        return []
        
    # 1. Dense Search
    query_dense = generate_multimodal_embedding(query_text, 'text')
    if query_dense is None:
        print("Failed to generate query embedding.")
        return []
        
    dense_scores = []
    for item in valid_chunks:
        if 'dense_embedding' in item and item['dense_embedding'] is not None:
            # Calculate cosine similarity manually with numpy
            a = np.array(query_dense)
            b = np.array(item['dense_embedding'])
            sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
            dense_scores.append(sim)
        else:
            dense_scores.append(0.0)
            
    # Normalize dense scores to [0, 1]
    dense_scores = np.array(dense_scores)
    if dense_scores.max() != dense_scores.min():
        dense_scores = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min())
        
    # 2. Sparse Search (BM25)
    tokenized_query = tokenize(query_text)
    sparse_scores = bm25.get_scores(tokenized_query)
    
    # Normalize sparse scores to [0, 1]
    sparse_scores = np.array(sparse_scores)
    if sparse_scores.max() != sparse_scores.min():
        sparse_scores = (sparse_scores - sparse_scores.min()) / (sparse_scores.max() - sparse_scores.min())
        
    # 3. Combine Scores
    combined_scores = alpha * dense_scores + (1 - alpha) * sparse_scores
    
    # Rank results
    ranked_indices = np.argsort(combined_scores)[::-1][:top_k]
    
    results = []
    for idx in ranked_indices:
        results.append({
            "chunk": valid_chunks[idx],
            "score": combined_scores[idx],
            "dense_score": dense_scores[idx],
            "sparse_score": sparse_scores[idx]
        })
        
    return results

# Test Hybrid Search and Display Results
if 'valid_chunks' in locals() and valid_chunks:
    query = "sample query" # Change this to test different queries
    print(f"Running hybrid search for: '{query}'")
    results = hybrid_search(query, alpha=0.7, top_k=3)

    for i, res in enumerate(results):
        chunk = res['chunk']
        print(f"\nResult {i+1}: Score: {res['score']:.3f} (Dense: {res['dense_score']:.3f}, Sparse: {res['sparse_score']:.3f})")
        print(f"Chunk ID: {chunk['chunk_id']}, Time: {chunk['start_time']:.1f}s - {chunk['end_time']:.1f}s")
        print(f"Desc: {chunk['description'][:100]}...")
        
        # Play video chunk
        local_path = chunk['file_path']
        if os.path.exists(local_path):
            try:
                with open(local_path, 'rb') as f:
                    video_bytes = f.read()
                encoded_video = base64.b64encode(video_bytes).decode('utf-8')
                video_html = f'''
                <video width="300" controls>
                    <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
                    Your browser does not support the video tag.
                </video>
                '''
                display(HTML(video_html))
            except Exception as e:
                print(f"Error displaying video: {e}")
        else:
            print(f"File not found: {local_path}")
else:
    print("Skipping search test as no valid chunks are available.")

## Phase 2d: Integrating Flickr 8k (Images) and Videos

In this phase, we integrate a more complex dataset:
1.  **Flickr 8k**: Images with professional annotations (`captions.txt`).
2.  **4 Long Videos**: We will chunk them and generate descriptions.

We will build a unified hybrid search index across both datasets.

In [ ]:
import pandas as pd
from google.cloud import storage
import io

# Read captions.txt from GCS
def read_gcs_file(bucket_name, blob_name):
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(blob_name)
    return blob.download_as_text()

bucket_name = "ai-multimodal-data"
blob_name = "Flickr 8k/captions.txt"

try:
    print(f"Reading {blob_name} from bucket {bucket_name}...")
    captions_text = read_gcs_file(bucket_name, blob_name)
    # Parse CSV text
    df_captions = pd.read_csv(io.StringIO(captions_text))
    print(f"Loaded {len(df_captions)} captions.")
    print(df_captions.head())
    
    # Combine captions per image
    image_captions = {}
    for index, row in df_captions.iterrows():
        img_name = row['image']
        caption = row['caption']
        if img_name not in image_captions:
            image_captions[img_name] = []
        image_captions[img_name].append(caption)
        
    # Join captions with space
    image_docs = {k: " ".join(v) for k, v in image_captions.items()}
    print(f"Combined into {len(image_docs)} image documents for BM25.")
    
except Exception as e:
    print(f"Error reading captions: {e}")

In [ ]:
# List of videos to process
videos = [
    "Eurasia_KBS_130903.mp4",
    "Finland_band_KBS_180929.mp4",
    "Pyeongyang_KBS_180915.mp4",
    "newyork_trip_KBS_170218.mp4"
]

# We will chunk them and generate descriptions for a subset
# Reuse chunk_video_opencv and generate_chunk_description functions

all_video_chunks = []
bucket_name = "ai-multimodal-data"

for vid in videos:
    gcs_uri = f"gs://{bucket_name}/{vid}"
    local_path = f"temp_{vid}"
    
    # Download if not exists
    if not os.path.exists(local_path):
        print(f"Downloading {gcs_uri}...")
        try:
            download_blob(bucket_name, vid, local_path)
        except Exception as e:
            print(f"Error downloading {vid}: {e}")
            continue
    else:
        print(f"Local file {local_path} already exists.")
            
    print(f"Chunking {vid}...")
    # Output to a unique folder per video
    output_folder = f"chunks_{vid.split('.')[0]}"
    chunks = chunk_video_opencv(local_path, 10, output_folder)
    
    # Generate descriptions and embeddings for top 5 chunks
    print(f"Generating descriptions and embeddings for top 5 chunks of {vid}...")
    for chunk in chunks[:5]:
        print(f"Processing {chunk['file_path']}...")
        # generate_chunk_description uses gemini-2.5-flash (or fallback)
        desc = generate_chunk_description(chunk['file_path'])
        chunk['description'] = desc
        chunk['source_video'] = vid
        
        # Generate dense embedding
        embedding = generate_multimodal_embedding(chunk['file_path'], 'video')
        chunk['dense_embedding'] = embedding
        
        all_video_chunks.append(chunk)

print(f"Processed {len(all_video_chunks)} video chunks in total.")

In [ ]:
import pickle
import os
from google.cloud import storage

combined_corpus = []
combined_tokenized = []
combined_registry = []
bucket_name = "ai-multimodal-data"
registry_file = 'full_dataset_registry.pkl'

# TBD: Replace with the actual GCS URI for the pre-computed registry
# Example: "gs://your-workshop-bucket/full_dataset_registry.pkl"
PRECOMPUTED_GCS_URI = "gs://ai-multimodal-data/TBD/full_dataset_registry.pkl" 

def download_precomputed_data(gcs_uri, local_path):
    """Downloads pre-computed data from GCS."""
    try:
        if "TBD" in gcs_uri:
            print("Skipping GCS download: Please update PRECOMPUTED_GCS_URI with the actual path if you want to use pre-computed data.")
            return False
            
        print(f"Attempting to download pre-computed data from {gcs_uri}...")
        # Extract bucket and blob name
        parts = gcs_uri.replace("gs://", "").split("/", 1)
        b_name = parts[0]
        blob_name = parts[1]
        
        storage_client = storage.Client()
        bucket = storage_client.bucket(b_name)
        blob = bucket.blob(blob_name)
        blob.download_to_filename(local_path)
        print(f"Successfully downloaded to {local_path}")
        return True
    except Exception as e:
        print(f"Failed to download from GCS: {e}")
        return False

# 1. Check if file exists locally, if not try to download from GCS
if not os.path.exists(registry_file):
    download_precomputed_data(PRECOMPUTED_GCS_URI, registry_file)

# 2. Load data if available
if os.path.exists(registry_file):
    print(f"Loading registry from {registry_file}...")
    try:
        with open(registry_file, 'rb') as f:
            combined_registry = pickle.load(f)
        print(f"Loaded {len(combined_registry)} items.")
        
        # Prepare corpus and tokenized corpus for BM25
        for item in combined_registry:
            if 'description' in item and item['description']:
                combined_corpus.append(item['description'])
                combined_tokenized.append(tokenize(item['description']))
                
    except Exception as e:
        print(f"Error loading registry file: {e}")
        print("Falling back to subset processing.")
        os.remove(registry_file) # Remove corrupted file if any
            
# 3. Fallback to subset processing if no data loaded
if not combined_registry:
    print(f"\nPre-computed data not available. Running subset processing for workshop practice...")
    
    # 1. Add Images (Flickr 8k)
    print("Processing images for registry...")
    IMAGE_LIMIT = 50
    count = 0

    if 'image_docs' in locals():
        for img_name, doc in image_docs.items():
            if count >= IMAGE_LIMIT:
                break
            
            gcs_path = f"gs://{bucket_name}/Flickr 8k/Images/{img_name}"
            embedding = generate_multimodal_embedding(gcs_path, 'image')
            
            if embedding is not None:
                combined_corpus.append(doc)
                combined_tokenized.append(tokenize(doc))
                combined_registry.append({
                    'id': img_name,
                    'type': 'image',
                    'path': gcs_path,
                    'description': doc,
                    'dense_embedding': embedding
                })
                count += 1
        print(f"Added {count} images to registry.")
    else:
        print("No image documents available.")
        
    # 2. Add Video Chunks
    print("\nProcessing video chunks for registry...")
    if 'all_video_chunks' in locals():
        for chunk in all_video_chunks:
            if 'description' in chunk and chunk['description'] and chunk['dense_embedding'] is not None:
                combined_corpus.append(chunk['description'])
                combined_tokenized.append(tokenize(chunk['description']))
                combined_registry.append({
                    'id': f"{chunk['source_video']}_{chunk['chunk_id']}",
                    'type': 'video_chunk',
                    'path': chunk['file_path'],
                    'description': chunk['description'],
                    'dense_embedding': chunk['dense_embedding']
                })
        print(f"Added {len(all_video_chunks)} video chunks to registry.")
    else:
        print("No video chunks available.")

# Build BM25 on combined corpus
if combined_tokenized:
    combined_bm25 = SimpleBM25(combined_tokenized)
    print(f"\nCombined BM25 index created with {len(combined_corpus)} documents.")
else:
    print("\nNo data for BM25.")

In [ ]:
def combined_hybrid_search(query_text, alpha=0.7, top_k=5):
    """Performs hybrid search across combined images and video chunks."""
    # FIX: Use globals() instead of locals() to check for the registry defined in the notebook scope
    if 'combined_registry' not in globals() or not combined_registry:
        print("No indexed items available.")
        return []
        
    # 1. Dense Search
    query_dense = generate_multimodal_embedding(query_text, 'text')
    if query_dense is None:
        print("Failed to generate query embedding.")
        return []
        
    dense_scores = []
    for item in combined_registry:
        if 'dense_embedding' in item and item['dense_embedding'] is not None:
            # Calculate cosine similarity manually with numpy
            a = np.array(query_dense)
            b = np.array(item['dense_embedding'])
            sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
            dense_scores.append(sim)
        else:
            dense_scores.append(0.0)
            
    dense_scores = np.array(dense_scores)
    if dense_scores.max() != dense_scores.min():
        dense_scores = (dense_scores - dense_scores.min()) / (dense_scores.max() - dense_scores.min())
        
    # 2. Sparse Search (BM25)
    tokenized_query = tokenize(query_text)
    sparse_scores = combined_bm25.get_scores(tokenized_query)
    
    sparse_scores = np.array(sparse_scores)
    if sparse_scores.max() != sparse_scores.min():
        sparse_scores = (sparse_scores - sparse_scores.min()) / (sparse_scores.max() - sparse_scores.min())
        
    # 3. Combine Scores
    combined_scores = alpha * dense_scores + (1 - alpha) * sparse_scores
    
    # Rank results
    ranked_indices = np.argsort(combined_scores)[::-1][:top_k]
    
    results = []
    for idx in ranked_indices:
        results.append({
            "item": combined_registry[idx],
            "score": combined_scores[idx],
            "dense_score": dense_scores[idx],
            "sparse_score": sparse_scores[idx]
        })
        
    return results

# Test combined search
if 'combined_registry' in locals() or 'combined_registry' in globals():
    query = "child in pink dress"
    print(f"Searching for: '{query}'")
    results = combined_hybrid_search(query, alpha=0.7)

    for i, res in enumerate(results):
        item = res['item']
        print(f"\nResult {i+1}: Score: {res['score']:.3f} (Dense: {res['dense_score']:.3f}, Sparse: {res['sparse_score']:.3f})")
        print(f"Type: {item['type']}, ID: {item['id']}")
        print(f"Desc: {item['description'][:100]}...")
else:
    print("Skipping search test as no combined registry is available.")

## Phase 3: Post-Search Optimization

In this phase, we optimize the results returned by the hybrid search to ensure relevance and diversity.

### Video Crowding
We implement a local version of "Video Crowding" to prevent results from being dominated by a single video source.

### Reranking
We add the code structure to call the **Vertex AI Ranking API** (or similar managed reranker) for high-precision reranking.

In [ ]:
def apply_video_crowding(search_results, max_per_video=2):
    """Filters search results to limit the number of chunks from the same video.
    
    Args:
        search_results: List of results from combined_hybrid_search.
        max_per_video: Maximum number of chunks allowed per video source.
    """
    filtered_results = []
    video_counts = {}
    
    for res in search_results:
        item = res['item']
        if item['type'] == 'video_chunk':
            # Extract video ID or name
            # ID format was f"{vid}_{chunk['chunk_id']}"
            video_id = item['id'].split('_')[0] 
            
            count = video_counts.get(video_id, 0)
            if count < max_per_video:
                filtered_results.append(res)
                video_counts[video_id] = count + 1
            else:
                print(f"Crowding: Skipping chunk {item['id']} from video {video_id} (limit reached)")
        else:
            # Images are not crowded in this implementation
            filtered_results.append(res)
            
    return filtered_results

# Test Crowding
if 'results' in locals() and results:
    print("Applying video crowding (max 1 per video for demo)...")
    crowded_results = apply_video_crowding(results, max_per_video=1)
    print(f"\nResults after crowding: {len(crowded_results)}")
    for i, res in enumerate(crowded_results):
        item = res['item']
        print(f"Result {i+1}: {item['id']} (Score: {res['score']:.3f})")
else:
    print("Skipping crowding test as no search results are available.")

In [ ]:
# Code structure for Vertex AI Ranking API (Reranking)
# Note: This is a conceptual implementation as actual API access may vary by project.

def rerank_results(query, results):
    """Conceptual function for calling Vertex AI Ranking API for re-scoring."""
    print(f"Calling Vertex AI Ranking API for query: '{query}'")
    print(f"Passing {len(results)} candidates for reranking...")
    
    # The latest way on GCP to implement re-ranking typically involves:
    # 1. Using the Vertex AI Search / Agent Builder Ranking API.
    # 2. Or deploying a custom cross-encoder model on a Vertex AI Endpoint.
    
    # Conceptual structure for Vertex AI Search Ranking API:
    """
    from google.cloud import discoveryengine_v1 as discoveryengine
    
    client = discoveryengine.RankServiceClient()
    
    # Define the request
    request = discoveryengine.RankRequest(
        ranking_config="projects/{PROJECT_ID}/locations/global/rankingConfigs/default_ranking_config",
        model="default", # or a specific model
        query=query,
        records=[
            discoveryengine.RankRecord(
                id=r['item']['id'],
                title=r['item'].get('type', ''),
                content=r['item'].get('description', '')
            ) for r in results
        ]
    )
    
    response = client.rank(request)
    
    # Map scores back to results
    # ...
    """
    
    print("\nReranking completed (Simulated). Returning results in original order.")
    return results

# Test Reranking
if 'crowded_results' in locals() and crowded_results:
    final_results = rerank_results(query, crowded_results)
else:
    print("Skipping reranking test as no crowded results are available.")

### 💡 Code Efficiency & Scalability Tips for Workshop

For this workshop practice, we have implemented several optimizations to keep execution fast and cost-effective:
1.  **Local Caching**: Videos are only downloaded if they do not already exist locally.
2.  **Data Subsetting**: We are processing only a small subset of images (50) and video chunks (5 per video) to avoid long wait times and high API costs during the practice.

**Production Scalability Considerations:**
In a real-world enterprise scenario, you should consider:
*   **Parallel Processing**: Use Python's `concurrent.futures` to call Gemini APIs in parallel for generating descriptions and embeddings, significantly reducing processing time.
*   **Batch APIs**: Utilize batch processing capabilities for embeddings if supported by the SDK for your content type.
*   **Distributed Processing**: For massive datasets, use tools like Dataflow or Ray to distribute the chunking and embedding workload across multiple nodes.

## Phase 4: Vector Search Implementation (Vertex AI)

In this phase, we will use **Vertex AI Vector Search** (formerly known as Matching Engine) to build a managed vector index. This allows us to perform fast and scalable similarity search on our multimodal embeddings.

### Phase 4a: Provisioning Index & Endpoint

> [!IMPORTANT]
> **Time Commitment:** Creating and deploying a Vector Search index is a heavy infrastructure operation and typically takes **45 to 60 minutes**. 
> For the purpose of this hands-on tutorial, we recommend connecting to an already existing index if available, or being prepared for this wait time. We use `STREAM_UPDATE` to allow real-time upserts once the index is deployed.

In [ ]:
# Provisioning Vector Search Index & Endpoint
INDEX_DISPLAY_NAME = "multimodal_grocery_index"
ENDPOINT_DISPLAY_NAME = "multimodal_grocery_endpoint"
DEPLOYED_INDEX_ID = "multimodal_grocery_deployed"

print("Connecting to Vector Search Index...")
indices = aiplatform.MatchingEngineIndex.list(filter=f'display_name="{INDEX_DISPLAY_NAME}"')
if indices:
    my_index = indices[0]
    print(f"Connected to Index: {my_index.resource_name}")
else:
    print("Index not found. Please ensure index_assets.py is running.")

print("Connecting to Index Endpoint...")
endpoints = aiplatform.MatchingEngineIndexEndpoint.list(filter=f'display_name="{ENDPOINT_DISPLAY_NAME}"')
if endpoints:
    my_index_endpoint = endpoints[0]
    print(f"Connected to Endpoint: {my_index_endpoint.resource_name}")
else:
    print("Endpoint not found.")

In [ ]:
# Option 1: Create a new PUBLIC Endpoint and deploy the index to it
# This is needed because Colab cannot access private endpoints directly.

if 'my_index' not in globals():
    print("Error: 'my_index' is not defined. Please make sure you have created or connected to an index in the previous cell.")
else:
    NEW_ENDPOINT_DISPLAY_NAME = ENDPOINT_DISPLAY_NAME + "_public"
    NEW_DEPLOYED_INDEX_ID = DEPLOYED_INDEX_ID + "_public"

    print(f"Creating new PUBLIC Endpoint: {NEW_ENDPOINT_DISPLAY_NAME}...")
    try:
        # Check if it already exists first
        endpoints = aiplatform.MatchingEngineIndexEndpoint.list(filter=f'display_name="{NEW_ENDPOINT_DISPLAY_NAME}"')
        if endpoints:
            my_index_endpoint = endpoints[0]
            print(f"Connected to existing Public Endpoint: {my_index_endpoint.resource_name}")
        else:
            my_index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
                display_name=NEW_ENDPOINT_DISPLAY_NAME,
                public_endpoint_enabled=True
            )
            print(f"Created Public Endpoint: {my_index_endpoint.resource_name}")
            
        # Check if index is already deployed to it
        deployed = False
        for d_index in my_index_endpoint.deployed_indexes:
            if d_index.id == NEW_DEPLOYED_INDEX_ID:
                deployed = True
                print(f"Index already deployed with ID: {NEW_DEPLOYED_INDEX_ID}")
                break
                
        if not deployed:
            print(f"Deploying index {my_index.resource_name} to public endpoint...")
            print("This operation typically takes 15-30 minutes.")
            my_index_endpoint.deploy_index(
                index=my_index,
                deployed_index_id=NEW_DEPLOYED_INDEX_ID
            )
            print(f"Index deployed successfully.")
            
        # Update DEPLOYED_INDEX_ID for search cells
        DEPLOYED_INDEX_ID = NEW_DEPLOYED_INDEX_ID
        print(f"Updated DEPLOYED_INDEX_ID to: {DEPLOYED_INDEX_ID}")

    except Exception as e:
        print(f"Error in Option 1 setup: {e}")
        print("Please ensure you have necessary permissions and the index exists.")

### Phase 4b: Indexing and Searching

Now we will upsert our embeddings into the managed index and implement the search function.

In [ ]:
from google.cloud.aiplatform_v1.types import IndexDatapoint
from google.cloud import aiplatform
import numpy as np

# Phase 4b: Indexing and Searching

# Reload the endpoint to ensure full initialization for querying
try:
    print("Reloading Index Endpoint for querying...")
    my_index_endpoint = aiplatform.MatchingEngineIndexEndpoint(my_index_endpoint.resource_name)
    print(f"Reloaded Endpoint: {my_index_endpoint.resource_name}")
except Exception as e:
    print(f"Failed to reload endpoint: {e}")
    print("Proceeding with existing endpoint instance.")

# Now we will upsert the embeddings we generated in Phase 2 into the managed Vector Search index.
# We use combined_registry instead of embeddings_registry.

print("Upserting embeddings to Vector Search...")
datapoints = []
if 'combined_registry' in globals() and combined_registry:
    for item in combined_registry:
        if 'dense_embedding' in item and item['dense_embedding'] is not None:
            # Sanitize the ID for Vector Search (alphanumeric and underscores only)
            dp_id = item['id'].replace(".", "_").replace("-", "_").replace(" ", "_")
            datapoints.append(
                IndexDatapoint(
                    datapoint_id=dp_id,
                    feature_vector=item['dense_embedding']
                )
            )
else:
    print("No combined_registry available in memory. Please load or generate it first.")

# This call upserts the datapoints to the index. 
# Since we used STREAM_UPDATE, they should be available for search shortly.
if datapoints:
    try:
        my_index.upsert_datapoints(datapoints=datapoints)
        print(f"Upserted {len(datapoints)} datapoints to Vertex AI Vector Search.")
    except Exception as e:
        print(f"Failed to upsert to Vertex AI: {e}")
else:
    print("No datapoints to upsert.")

def find_similar_content(query_input, query_type, top_k=5):
    """Searches for similar content using Vertex AI Vector Search.
    
    Args:
        query_input: The text query or file path for image/video queries.
        query_type: 'text', 'image', or 'video'.
        top_k: Number of neighbors to return.
    """
    # Generate embedding for the query
    if query_type == 'text':
        query_embedding = generate_multimodal_embedding(query_input, 'text')
    elif query_type in ['image', 'video']:
        query_embedding = generate_multimodal_embedding(query_input, query_type)
    else:
        return []

    if query_embedding is None:
        return []

    # Perform the Vector Search query using the SDK
    try:
        response = my_index_endpoint.find_neighbors(
            queries=[query_embedding],
            num_neighbors=top_k,
            deployed_index_id=DEPLOYED_INDEX_ID
        )

        # Process the results and map back to our local registry
        results = []
        for neighbor in response[0]:
            # Find item in registry by comparing sanitized IDs
            item = next((x for x in combined_registry if x['id'].replace(".", "_").replace("-", "_").replace(" ", "_") == neighbor.id), None)
            if item:
                results.append({
                    'id': item['id'],
                    'score': neighbor.distance,
                    'type': item['type'],
                    'path': item['path']
                })
        return results
    except Exception as e:
        print(f"Search failed: {e}")
        return []

# Test search with a text query
if datapoints:
    print("\nTesting search with text query: 'lemons'")
    try:
        test_results = find_similar_content('lemons', 'text', top_k=3)
        for res in test_results:
            print(f"Found Match: {res['id']} (Score: {res['score']:.4f})")
    except Exception as e:
        print(f"Search test failed: {e}")

## Phase 5: Interactive Visualization

In this phase, we will create a simple UI using `ipywidgets` to allow users to query the system with text, images, or videos and see the results.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage, HTML
import random
import os
import base64

# UI Components
query_type_dropdown = widgets.Dropdown(
    options=['text', 'image', 'video'],
    value='text',
    description='Query Type:',
)

text_query_input = widgets.Text(
    value='red',
    placeholder='Enter text query',
    description='Text Query:',
    disabled=False
)

upload_button = widgets.FileUpload(
    accept='image/*,video/*',
    multiple=False,
    description='Upload File'
)

shuffle_button = widgets.Button(
    description='Shuffle Gallery Query',
    button_style='info'
)

search_button = widgets.Button(
    description='Search',
    button_style='success'
)

output_area = widgets.Output()

def display_media_results(results):
    """Helper function to display results with improved layout and deduplication."""
    
    # Deduplicate results based on ID to prevent double rendering
    seen_ids = set()
    deduped_results = []
    for res in results:
        if res['id'] not in seen_ids:
            deduped_results.append(res)
            seen_ids.add(res['id'])
            
    images = [res for res in deduped_results if res['type'] == 'image']
    videos = [res for res in deduped_results if res['type'] in ['video', 'video_chunk']]
    
    print(f"\nTop Results (Total: {len(deduped_results)} after deduplication):")
    
    # --- Enforce counts ---
    # For Images: Take up to 3 from results
    display_images = images[:3]
    # If fewer than 3, find more in registry
    if len(display_images) < 3:
        found_ids = {res['id'] for res in deduped_results}
        if 'combined_registry' in globals():
            for item in combined_registry:
                if item['type'] == 'image' and item['id'] not in found_ids:
                    display_images.append({
                        'id': item['id'],
                        'score': 0.0, # Unknown score for fallback
                        'type': 'image',
                        'path': item['path']
                    })
                    found_ids.add(item['id'])
                    if len(display_images) >= 3:
                        break
                    
    # For Video: Take first from results
    display_videos = videos[:1]
    # If none, find first in registry
    if not display_videos:
        if 'combined_registry' in globals():
            for item in combined_registry:
                if item['type'] in ['video', 'video_chunk']:
                    display_videos.append({
                        'id': item['id'],
                        'score': 0.0,
                        'type': item['type'],
                        'path': item['path']
                    })
                    break
    # ----------------------
    
    # Display Images (exactly 3 if available)
    img_widgets = []
    for img in display_images[:3]:
        out = widgets.Output()
        with out:
            if img['score'] > 0:
                print(f"Score: {img['score']:.4f} | {img['id']}")
            else:
                print(f"Gallery Fallback | {img['id']}")
                
            path = img['path']
            try:
                if path.startswith("gs://"):
                    from google.cloud import storage
                    storage_client = storage.Client()
                    parts = path[5:].split('/', 1)
                    b_name = parts[0]
                    bl_name = parts[1]
                    bucket = storage_client.bucket(b_name)
                    blob = bucket.blob(bl_name)
                    image_bytes = blob.download_as_bytes()
                    display(IPImage(data=image_bytes, width=250))
                else:
                    display(IPImage(filename=path, width=250))
            except Exception as e:
                print(f"Error displaying image: {e}")
                
        img_widgets.append(out)
        
    if img_widgets:
        print("\nImage Previews:")
        display(widgets.HBox(img_widgets))
        
    # Display Video (exactly 1 if available)
    if display_videos:
        v = display_videos[0]
        if v['score'] > 0:
            print(f"\nVideo Preview - Score: {v['score']:.4f} | {v['id']}")
        else:
            print(f"\nVideo Preview - Gallery Fallback | {v['id']}")
            
        path = v['path']
        try:
            if path.startswith("gs://"):
                from google.cloud import storage
                storage_client = storage.Client()
                parts = path[5:].split('/', 1)
                b_name = parts[0]
                bl_name = parts[1]
                bucket = storage_client.bucket(b_name)
                blob = bucket.blob(bl_name)
                video_bytes = blob.download_as_bytes()
            else:
                with open(path, 'rb') as f:
                    video_bytes = f.read()
                    
            encoded_video = base64.b64encode(video_bytes).decode('utf-8')
            video_html = f'''
            <video width="400" controls>
                <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
                Your browser does not support the video tag.
            </video>
            '''
            display(HTML(video_html))
        except Exception as e:
            print(f"Error displaying video: {e}")
            
    else:
        print("No video found in gallery.")

def on_search_clicked(b):
    with output_area:
        # FIX: Use wait=True to prevent appending outputs if called multiple times
        clear_output(wait=True)
        q_type = query_type_dropdown.value
        
        if q_type == 'text':
            query = text_query_input.value
            print(f"Searching for text: '{query}'")
            results = find_similar_content(query, 'text', top_k=30)
        elif q_type in ['image', 'video']:
            if upload_button.value:
                uploaded_files = upload_button.value
                if isinstance(uploaded_files, list):
                    uploaded_file = uploaded_files[0]
                    content = uploaded_file['content']
                    filename = uploaded_file['name']
                else:
                    filename = list(uploaded_files.keys())[0]
                    content = uploaded_files[filename]['content']
                
                filepath = os.path.join(DATA_DIR, f"uploaded_{filename}")
                with open(filepath, 'wb') as f:
                    f.write(content)
                print(f"Searching with uploaded file: {filename}")
                results = find_similar_content(filepath, q_type, top_k=30)
            else:
                print("Please upload a file first for image/video queries.")
                return
                
        display_media_results(results)

def on_shuffle_clicked(b):
    if 'combined_registry' not in globals() or not combined_registry:
        print("No embeddings in registry.")
        return
    
    random_item = random.choice(combined_registry)
    random_file = random_item['id']
    path = random_item['path']
    
    with output_area:
        # FIX: Use wait=True
        clear_output(wait=True)
        print(f"Shuffled Query: {random_file}")
        
        try:
            if path.startswith("gs://"):
                from google.cloud import storage
                storage_client = storage.Client()
                parts = path[5:].split('/', 1)
                b_name = parts[0]
                bl_name = parts[1]
                bucket = storage_client.bucket(b_name)
                blob = bucket.blob(bl_name)
                media_bytes = blob.download_as_bytes()
                
                if random_item['type'] == 'image':
                    display(IPImage(data=media_bytes, width=200))
                    results = find_similar_content(path, 'image', top_k=30)
                elif random_item['type'] in ['video', 'video_chunk']:
                    encoded_video = base64.b64encode(media_bytes).decode('utf-8')
                    video_html = f'''
                    <video width="300" controls>
                        <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
                        Your browser does not support the video tag.
                    </video>
                    '''
                    display(HTML(video_html))
                    results = find_similar_content(path, 'video', top_k=30)
            else:
                print(f"Local file paths are not supported in this shuffle function: {path}")
                return
                
            display_media_results(results)
            
        except Exception as e:
            print(f"Error handling shuffled query: {e}")

search_button.on_click(on_search_clicked)
shuffle_button.on_click(on_shuffle_clicked)

# Layout
input_box = widgets.VBox([
    query_type_dropdown,
    text_query_input,
    upload_button,
    widgets.HBox([search_button, shuffle_button])
])

display(input_box, output_area)

with output_area:
    if 'test_results' in globals() and test_results:
        print("Displaying results from the previous test search ('lemons'):")
        display_media_results(test_results)

## Phase 6: Audio & Video as Search Queries

Gemini Embedding 2.0 allows you to use audio or video files directly as queries to find similar content in your index. This enables "query-by-example" workflows.

In [ ]:
# 1. Create a dummy audio file (silent beep) for the demonstration
import wave
import struct

AUDIO_FILE = 'sample_audio.mp3'
# Note: Creating a simple WAV and naming it mp3 for the sake of the demo's mime-type
with wave.open(AUDIO_FILE, 'w') as f:
    f.setnchannels(1)
    f.setsampwidth(2)
    f.setframerate(44100)
    for i in range(44100):
        value = int(32767.0 * 0.5)
        data = struct.pack('<h', value)
        f.writeframesraw(data)

print(f"Generated dummy audio: {AUDIO_FILE}")

In [ ]:
# 2. Search using the Audio file as a query
print(f"Searching for content similar to audio: {AUDIO_FILE}...")
audio_results = find_similar_content(AUDIO_FILE, 'audio', top_k=2)

for res in audio_results:
    print(f"Found Match: {res['id']} (Score: {res['score']:.4f})")
    if res['type'] == 'image':
        display(IPImage(filename=res['path'], width=150))
    elif res['type'] == 'video':
        display(IPVideo(filename=res['path'], embed=True, width=150))

In [ ]:
# Phase 6: Video Generation & Preview
from google import genai
from google.genai import types
from IPython.display import display, HTML
import base64
import time

PROMPT = "a person scooping ice cream"
QUERY_VIDEO = 'ice_cream_scoop.mp4'
VIDEO_MODEL = "veo-3.1-generate-001"

print(f"Attempting to generate video with prompt: '{PROMPT}' using {VIDEO_MODEL}...")

try:
    # Cell 4 initialized `client` with `vertexai=True`. We can use it directly.
    if 'client' not in globals():
         print("Initializing GenAI client...")
         client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)
         
    print("Starting video generation operation (this may take several minutes)...")
    operation = client.models.generate_videos(
        model=VIDEO_MODEL,
        prompt=PROMPT,
    )
    
    # Poll for completion
    while not operation.done:
        print("  Waiting for video generation...")
        time.sleep(20)
        operation = client.operations.get(operation)
        
    generated_video = operation.response.generated_videos[0]
    
    # Save the video
    generated_video.video.save(QUERY_VIDEO)
    print(f"Video generated and saved to {QUERY_VIDEO}")
    
    # Preview
    print("\nPreviewing generated video:")
    with open(QUERY_VIDEO, 'rb') as f:
        video_bytes = f.read()
    encoded_video = base64.b64encode(video_bytes).decode('utf-8')
    video_html = f'''
    <video width="400" controls>
        <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    '''
    display(HTML(video_html))

except Exception as e:
    print(f"Video generation failed: {e}")
    print("Falling back to dummy video generation with OpenCV for the sake of the tutorial flow.")
    
    # Fallback to OpenCV generation so the tutorial doesn't break completely
    import cv2
    import numpy as np
    
    width, height = 400, 400
    fps = 30
    duration = 4
    total_frames = fps * duration
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(QUERY_VIDEO, fourcc, fps, (width, height))
    for i in range(total_frames):
        frame = np.ones((height, width, 3), dtype=np.uint8) * 220
        frame[:, :, 0] = 255
        cv2.circle(frame, (200, 250), 80, (255, 255, 255), -1)
        if i < 60:
            spoon_y = 50 + int(i * 150 / 60)
            cv2.line(frame, (200, 50), (200, spoon_y), (100, 100, 100), 10)
            cv2.circle(frame, (200, spoon_y), 20, (100, 100, 100), -1)
        else:
            spoon_y = 200 - int((i - 60) * 150 / 60)
            cv2.line(frame, (200, 50), (200, spoon_y), (100, 100, 100), 10)
            cv2.circle(frame, (200, spoon_y), 20, (100, 100, 100), -1)
            cv2.circle(frame, (200, spoon_y - 10), 15, (255, 255, 255), -1)
        out.write(frame)
    out.release()
    print(f"Fallback dummy video generated and saved to {QUERY_VIDEO}")
    
    # Preview fallback
    with open(QUERY_VIDEO, 'rb') as f:
        video_bytes = f.read()
    encoded_video = base64.b64encode(video_bytes).decode('utf-8')
    video_html = f'''
    <video width="300" controls>
        <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    '''
    display(HTML(video_html))

In [ ]:
# Phase 6: Search with Generated Video & Preview Results
print(f"Searching for content similar to video: {QUERY_VIDEO}...")

video_results = find_similar_content(QUERY_VIDEO, 'video', top_k=2)

for res in video_results:
    print(f"Found Match: {res['id']} (Score: {res['score']:.4f})")
    path = res['path']
    try:
        if path.startswith("gs://"):
            from google.cloud import storage
            storage_client = storage.Client()
            parts = path[5:].split('/', 1)
            b_name = parts[0]
            bl_name = parts[1]
            bucket = storage_client.bucket(b_name)
            blob = bucket.blob(bl_name)
            media_bytes = blob.download_as_bytes()
            
            if res['type'] == 'image':
                from IPython.display import Image as IPImage
                display(IPImage(data=media_bytes, width=150))
            elif res['type'] == 'video':
                import base64
                from IPython.display import HTML
                encoded_video = base64.b64encode(media_bytes).decode('utf-8')
                video_html = f'''
                <video width="150" controls>
                    <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
                    Your browser does not support the video tag.
                </video>
                '''
                display(HTML(video_html))
        else:
            if res['type'] == 'image':
                from IPython.display import Image as IPImage
                display(IPImage(filename=path, width=150))
            elif res['type'] == 'video':
                from IPython.display import Video as IPVideo
                display(IPVideo(filename=path, embed=True, width=150))
    except Exception as e:
        print(f"Error displaying media for {res['id']}: {e}")

In [ ]:
# Final Test Search
print("Searching for 'futuristic city'...")
results = find_similar_content("futuristic city", "text", top_k=3)

for res in results:
    print(f"Found: {res['id']} (Score: {res['score']:.4f})")
    path = res['path']
    try:
        if path.startswith("gs://"):
            from google.cloud import storage
            storage_client = storage.Client()
            parts = path[5:].split('/', 1)
            b_name = parts[0]
            bl_name = parts[1]
            bucket = storage_client.bucket(b_name)
            blob = bucket.blob(bl_name)
            media_bytes = blob.download_as_bytes()
            
            if res['type'] == 'image':
                from IPython.display import Image as IPImage
                display(IPImage(data=media_bytes, width=300))
                
            elif res['type'] == 'video':
                import base64
                from IPython.display import HTML
                encoded_video = base64.b64encode(media_bytes).decode('utf-8')
                video_html = f'''
                <video width="300" controls>
                    <source src="data:video/mp4;base64,{encoded_video}" type="video/mp4">
                    Your browser does not support the video tag.
                </video>
                '''
                display(HTML(video_html))
        else:
            if res['type'] == 'image':
                from IPython.display import Image as IPImage
                display(IPImage(filename=path, width=300))
            elif res['type'] == 'video':
                from IPython.display import Video as IPVideo
                display(IPVideo(filename=path, embed=True, width=300))
    except Exception as e:
        print(f"Error displaying media for {res['id']}: {e}")